# RSO-111: Analyze louver configurations

During the shutdown period (31.1.26 -14.2.26) we ran multiple versions (different louver configurations) of BLOCK-T679. This notebook creates a table and plot with said configurations. 

**Description**

Output a table with the different configurations of louvers used and a simple plot showing the timeline.

**Expected results:**

File: summary of the different configurations used

File:  list of experiment segments with start time and louver configuration and temperature, airflow telemetry for each configuration


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient

In [ ]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

# Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

In [ ]:
def query_telemetry(start, end, freq):
    # get the complete telemetry for the period start-end
    df_temp = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.temperature", 
        columns=["sensorName","temperatureItem0"],
        begin=start,
        end=end,
    )
    df_temp.index.name = "time_stamp"
    # resample to the selected frequency
    df_temp_resampled = (
        df_temp.groupby("sensorName")["temperatureItem0"].resample(freq).mean().reset_index()
    )
    # use a column per sensor in the output dataframe
    df_temp_pivot = df_temp_resampled.pivot(
        index="time_stamp",
        columns="sensorName",
        values="temperatureItem0"
    )
    #now the same for airflow
    df_air = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.airFlow", 
        columns=["location","direction","speed"],
        begin=start,
        end=end,
    )
    df_air.index.name = "time_stamp"
    df_air_resampled = (
        df_air.groupby("location")[["speed","direction"]].resample(freq).mean().reset_index()
    )
    df_air_pivot_speed = df_air_resampled.pivot(
        index="time_stamp",
        columns="location",
        values="speed"
    )
    df_air_pivot_direction = df_air_resampled.pivot(
        index="time_stamp",
        columns="location",
        values="direction"
    )
    df_air_pivot_speed = df_air_pivot_speed.add_suffix("_speed")
    df_air_pivot_direction = df_air_pivot_direction.add_suffix("_direction")
    df_air_pivot = df_air_pivot_speed.join(df_air_pivot_direction, how="inner")
    df_combined = df_temp_pivot.join(df_air_pivot, how="inner")
    
    return df_combined


In [ ]:
def query_telemetry_old(start, end, freq):
    # get the complete telemetry for the period start-end
    df_telemetry = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.temperature", 
        columns=["sensorName","temperatureItem0"],
        begin=start,
        end=end,
    )
    df_telemetry.index.name = "time_stamp"
    # resample to the selected frequency
    df_telemetry_resampled = (
        df_telemetry.groupby("sensorName")["temperatureItem0"].resample(freq).mean().reset_index()
    )
    #now the same for airflow
    
    # use a column per sensor in the output dataframe
    df_pivot = df_telemetry_resampled.pivot(
        index="time_stamp",
        columns="sensorName",
        values="temperatureItem0"
    )

    return df_pivot


# Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In the following cell, we create a file with a summary of all louver configurations in the period, grouped by type

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
with open("louver_configs_summary.txt", "w") as f:
    for _, row in combination_counts.iterrows():
    
        conf_id = row['louvers_conf']
        count = row['count']
        
        line = f"Configuration {conf_id} (appears {count} times):"
        f.write(str(line) + "\n")
        
        # Extract position values
        config = row[position_cols]
        
        # Keep only non-zero values
        non_zero = config[config != 0]
        
        if len(non_zero) == 0:
            line = "  All positions are 0"
            f.write(str(line) + "\n")
        else:
            for col, val in non_zero.items():
                line = f"  {col}: {val}"
                f.write(str(line) + "\n")
        
        line = "-" * 40
        f.write(str(line) + "\n")

let's add a duration field, just looking at the time stamps of successive configurations

In [ ]:
df_setlouvers['duration'] = (
    df_setlouvers['time_stamp'].shift(-1) - df_setlouvers['time_stamp']
)
df_setlouvers['duration_minutes'] = df_setlouvers['duration'].dt.total_seconds() / 60

Now we dump into a file for reference each time a louver configuration was set, 
with its time stamp and louver configuration id according to the summary table


In [ ]:
df_setlouvers.to_csv("louver_configuration_instances.csv", index=False)

loop over the dataframe and dump into a file a dataframe the temperatures, airflow data

In [ ]:
dfs = []
for i,row in enumerate(df_setlouvers.itertuples()):
    print(f"Acquiring data from test {i}")
    start = row.time_stamp
    end = start + row.duration
    if pd.isna(end): 
        end = t_end_period
    df_tel = query_telemetry(Time(start), Time(end), '1min')
    df_tel["test_id"] = row.Index
    df_tel["louvers_conf"] = row.louvers_conf
    
    dfs.append(df_tel)

df_telemetry_all = pd.concat(dfs)

In [ ]:
df_telemetry_all.to_csv("telemetry_instances.csv")